# calo-ddpm — cent4 DDNM box-size sweep (full diagnostic suite)

Analyzes the **cent4 DDNM** inpainting runs across dead-region **box sizes**
(the data collected in the previous session).  It ports the complete
per-geometry diagnostic suite from `calo_method_comparison.ipynb`
(diagnostics 1–9 + Part B shrinkage) and the scalar metric-vs-box sweep
from `calo_sweep_analysis.ipynb`, but with **box size as the compared
series** rather than algorithm.  The model-free baselines (`noise`,
`meanfill`) are overlaid at each box size as calibration/accuracy floors
wherever they are present.

Every series is keyed by `(method, box)`; runs are auto-discovered from the
cent4 study directory, truncated to their completed image count
(`progress.txt`) and NaN-guarded — no hardcoded `N_EVENTS`.

**Analysis space.** `SPACE = 'gev'` (default) or `'log'` (env `CALO_SPACE`).
The DDPM samples in log space $x=\ln E$; stored results are $e^{x}$, so
`log` recovers the raw DDPM output exactly.  Rank-based tests (PIT,
coverage) are invariant under this monotone map; moment-based tests
(z-scores, bias, CRPS, sharpness, Part-B regression) are the cleaner,
near-Gaussian versions in `log`.  Energy-sum physics observables
(diagnostics 1–3) always use GeV.  All stored quantities are GeV.

**Convention in every plot:** color encodes box size, marker/linestyle
encodes method (DDNM solid ●, noise dashed ■, meanfill dotted ▲).

In [ ]:
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm as MplLogNorm
from scipy import stats

DATA_ROOT   = os.environ.get('CALO_ROOT', 'workdir')       # repo-root relative
STUDY_GLOB  = os.environ.get(
    'CALO_CENT4_GLOB',
    f'{DATA_ROOT}/inpaint_study_cent4/*_box*_eta*_phi*_S*')
EVENTS_PATH = os.environ.get(
    'CALO_EVENTS', f'{DATA_ROOT}/generated_events/events_cent4_seed0.npy')

# which methods to include (DDNM is the subject; the rest are floors)
METHODS = os.environ.get('CALO_METHODS', 'ddnm,noise,meanfill').split(',')

# marker / linestyle per method; color is assigned per box size below
METHOD_STYLE = {'ddnm': ('-', 'o'), 'noise': ('--', 's'),
                'meanfill': (':', '^')}

SPACE = os.environ.get('CALO_SPACE', 'gev')     # 'gev' or 'log' (raw DDPM)
assert SPACE in ('gev', 'log')
UNIT = '[GeV]' if SPACE == 'gev' else '[ln GeV]'
print('analysis space for value-level diagnostics:', SPACE)
print('study glob :', STUDY_GLOB)
print('truth file :', EVENTS_PATH)

## Load runs (auto-discovered, progress-truncated, NaN-guarded)

Each run dir is one `(method, box)` point.  Series are ordered DDNM first (by box), then baselines.

In [ ]:
true_events = np.asarray(np.load(EVENTS_PATH, mmap_mode='r'))
print('truth events:', EVENTS_PATH, true_events.shape)

def slabel(method, box):
    return f'{method} box{box}'

# discover
records = []
for run_dir in sorted(glob.glob(STUDY_GLOB)):
    if not os.path.exists(f'{run_dir}/results.npy'):
        continue
    meta = json.load(open(f'{run_dir}/metadata.json'))
    if meta['algorithm'] not in METHODS:
        continue
    records.append((run_dir, meta))

boxes = sorted({m['box'] for _, m in records})
# color by box size (viridis); one consistent color per box across all plots
_cmap = plt.cm.viridis
BOX_COLOR = {b: _cmap(i / max(1, len(boxes) - 1))
             for i, b in enumerate(boxes)}
print('box sizes found:', boxes)

def series_style(r):
    ls, marker = METHOD_STYLE.get(r['method'], ('-', 'o'))
    return dict(color=BOX_COLOR[r['BOX']], ls=ls, marker=marker)

# build series (DDNM boxes first, then the other methods)
def _order(rec):
    _, m = rec
    prio = METHODS.index(m['algorithm']) if m['algorithm'] in METHODS else 99
    return (prio, m['box'])

series = {}
for run_dir, meta in sorted(records, key=_order):
    alg = meta['algorithm']
    res = np.load(f'{run_dir}/results.npy')          # (n, J, b, b)
    tru = np.load(f'{run_dir}/truth.npy')            # (n, b, b)

    done = res.shape[0]
    if os.path.exists(f'{run_dir}/progress.txt'):
        done = int(open(f'{run_dir}/progress.txt').read().split('/')[0])
    res, tru = res[:done], tru[:done]

    finite = np.isfinite(res).all(axis=tuple(range(1, res.ndim)))
    if (~finite).any():
        print(f'WARNING {alg} box{meta["box"]}: '
              f'{(~finite).sum()}/{done} non-finite images excluded')
    idx = np.where(finite)[0]
    res, tru = res[finite], tru[finite]
    if res.shape[0] == 0:
        print(f'skip {alg} box{meta["box"]}: no finite images')
        continue

    label = slabel(alg, meta['box'])
    series[label] = {
        'method'      : alg,
        'inpaint_dead': res[:, :, None].astype(np.float64),   # (n,J,1,b,b)
        'truth_dead'  : tru[:, None].astype(np.float64),       # (n,1,b,b)
        'idx'         : idx,
        'Y0': meta.get('eta0', meta.get('y0')),
        'X0': meta.get('phi0', meta.get('x0')),
        'BOX': int(meta['box']), 'meta': meta,
    }
    print(f'{label:16s} n={res.shape[0]:5d}  J={res.shape[1]}  '
          f'box={meta["box"]}  S={meta["S"]}')

# value-level views in the chosen analysis space
for r in series.values():
    if SPACE == 'log':
        r['val_dead']  = np.log(np.clip(r['inpaint_dead'], 1e-30, None))
        r['val_truth'] = np.log(np.clip(r['truth_dead'],  1e-30, None))
    else:
        r['val_dead']  = r['inpaint_dead']
        r['val_truth'] = r['truth_dead']

labels = list(series)
assert labels, 'no completed runs found under STUDY_GLOB'
ddnm_labels = [l for l in labels if series[l]['method'] == 'ddnm']
print('\nseries:', labels)

## Precompute per-image quantities (exact box decomposition)

Outside the dead region the reconstruction equals the truth, so full-image sums / dot-products decompose exactly into a fixed truth part plus the box contribution.

In [ ]:
for label, r in series.items():
    imgs = true_events[r['idx']].astype(np.float64)[:, None]   # (n,1,H,W)
    r['true_full_sum']  = imgs.sum(axis=(1, 2, 3))
    r['true_full_mean'] = imgs.mean(axis=(1, 2, 3))
    r['true_box_sum']   = r['truth_dead'].sum(axis=(1, 2, 3))
    r['reco_box_sum']   = r['inpaint_dead'].sum(axis=(2, 3, 4))
    r['reco_full_sum']  = (r['true_full_sum'][:, None]
                           - r['true_box_sum'][:, None] + r['reco_box_sum'])
    r['D']        = imgs[0].size
    r['dot_tt']   = (imgs ** 2).sum(axis=(1, 2, 3))
    r['dot_tbtb'] = (r['truth_dead'] ** 2).sum(axis=(1, 2, 3))
    r['dot_tbsb'] = (r['truth_dead'][:, None] * r['inpaint_dead']).sum(axis=(2, 3, 4))
    r['dot_sbsb'] = (r['inpaint_dead'] ** 2).sum(axis=(2, 3, 4))
    del imgs
print('precomputed for', len(series), 'series.')

## Diagnostic 1 — Missing energy ratio (accuracy)

True/reco missing (dead-region) energy vs the true missing energy, one series per (method, box).  Unity = unbiased recovery of the removed energy.  Larger boxes remove more energy and are harder to recover.

In [ ]:
def log_bins(vals, n):
    v = np.asarray(vals); v = v[np.isfinite(v) & (v > 0)]
    return np.geomspace(v.min() * 0.8, v.max() * 1.2, n)

def binned_weighted_stats(x, y, dy, bins):
    x, y, dy, bins = map(np.asarray, (x, y, dy, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan); cnt = np.zeros(nb, int)
    for i in range(nb):
        m = (idx == i) & (dy > 0)
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        w = 1.0 / dy[m] ** 2
        mean[i] = np.sum(w * y[m]) / w.sum()
        err[i]  = np.sqrt(1.0 / w.sum())
    return centers, mean, err, cnt

def binned_mean_stats(x, y, bins):
    x, y, bins = map(np.asarray, (x, y, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan); cnt = np.zeros(nb, int)
    for i in range(nb):
        m = idx == i
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        v = y[m]
        mean[i] = v.mean()
        err[i]  = v.std(ddof=1) / np.sqrt(cnt[i]) if cnt[i] > 1 else 0.0
    return centers, mean, err, cnt

bins = log_bins(np.concatenate([r['truth_dead'].mean(axis=(1, 2, 3))
                                for r in series.values()]), 26)
plt.figure(figsize=(7, 4.5))
for label, r in series.items():
    ratio = r['true_box_sum'][:, None] / np.clip(r['reco_box_sum'], 1e-30, None)
    centers, mean, sig, _ = binned_weighted_stats(
        r['truth_dead'].mean(axis=(1, 2, 3)),
        ratio.mean(axis=1), ratio.std(axis=1), bins)
    st = series_style(r)
    plt.errorbar(centers, mean, yerr=sig, fmt=st['marker'], color=st['color'],
                 ls='none', capsize=3, ms=4, label=label)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xscale('log')
plt.xlabel('average true missing energy [GeV]')
plt.ylabel('true / reco missing energy')
plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## Diagnostic 2 — Full-image reco/true energy ratio

Reconstructed vs true total event energy (the box error diluted over the whole image).  Deviations from unity shrink as the box shrinks.

In [ ]:
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in series.values()]), 41)
plt.figure(figsize=(7, 4.5))
for label, r in series.items():
    ratio = r['reco_full_sum'] / r['true_full_sum'][:, None]
    centers, mean, sig, _ = binned_weighted_stats(
        r['true_full_mean'], ratio.mean(axis=1), ratio.std(axis=1) + 1e-12,
        bins)
    st = series_style(r)
    plt.errorbar(centers, mean, yerr=sig, fmt=st['marker'], color=st['color'],
                 ls='none', capsize=3, ms=4, label=label)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xscale('log')
plt.xlabel('average true energy [GeV]')
plt.ylabel('average reco/true energy')
plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## Diagnostic 3 — Full-image Pearson (algebraic, exact)

Per-event Pearson correlation between reco and truth over all towers, computed algebraically from the exact box decomposition.

In [ ]:
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in series.values()]), 41)
plt.figure(figsize=(7, 4.5))
for label, r in series.items():
    D = r['D']
    t_mean = r['true_full_mean']
    s_mean = r['reco_full_sum'] / D
    dot_ts = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_tbsb'])
    dot_ss = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_sbsb'])
    cov = dot_ts - D * t_mean[:, None] * s_mean
    vt  = r['dot_tt'] - D * t_mean ** 2
    vs  = dot_ss - D * s_mean ** 2
    pearson = cov / np.sqrt(np.clip(vt[:, None] * vs, 1e-30, None))
    centers, mean, sig, _ = binned_mean_stats(t_mean, pearson.mean(axis=1), bins)
    st = series_style(r)
    plt.errorbar(centers, mean, yerr=sig, fmt=st['marker'], color=st['color'],
                 ls='none', capsize=3, ms=4, label=label)
plt.xscale('log')
plt.xlabel('average true energy [GeV]')
plt.ylabel('average full-image Pearson coefficient')
plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## Diagnostic 4 — z-score diagnostics (dead region)

Per-tower / per-event standardized residuals $(t-\mu)/\sigma$ of the posterior.  Calibrated targets: event-z mean 0, pixel RMS(z) 1, fraction $|z|<1$ ≈ 0.683.  Value-space diagnostic (uses `SPACE`).

In [ ]:
def zscore_diag(r):
    t, s = r['val_truth'], r['val_dead']
    mu = s.mean(axis=1); sd = s.std(axis=1, ddof=1)
    sd = np.where(sd < 1e-12, np.nan, sd)
    z = (t - mu) / sd
    bm_s = s.mean(axis=(2, 3, 4))
    ev_mu, ev_sd = bm_s.mean(axis=1), bm_s.std(axis=1, ddof=1)
    ev_sd = np.where(ev_sd < 1e-12, np.nan, ev_sd)
    ev_z = (t.mean(axis=(1, 2, 3)) - ev_mu) / ev_sd
    return {'event_z': ev_z,
            'rms_z': np.sqrt(np.nanmean(z**2, axis=(1, 2, 3))),
            'frac_lt1': np.nanmean(np.abs(z) < 1, axis=(1, 2, 3))}

zres = {label: zscore_diag(r) for label, r in series.items()}
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in series.values()]), 21)
for key, ylabel, ref in [('event_z', 'average z-score (box mean)', 0.0),
                         ('rms_z', 'pixel RMS(z)', 1.0),
                         ('frac_lt1', 'fraction |z| < 1', 0.6827)]:
    plt.figure(figsize=(7, 4.5))
    for label, res in zres.items():
        r = series[label]
        centers, mean, sig, _ = binned_mean_stats(r['true_full_mean'],
                                                  res[key], bins)
        st = series_style(r)
        plt.errorbar(centers, mean, yerr=sig, fmt=st['marker'],
                     color=st['color'], ls='none', capsize=3, ms=4, label=label)
    plt.axhline(ref, ls='--', color='k')
    plt.xscale('log')
    plt.xlabel('average true energy [GeV]'); plt.ylabel(ylabel)
    plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.show()

## Diagnostic 5 — Spatial bias maps

mean(inpainted − truth) per tower, one panel per series (note box sizes differ, so panels have different pixel extents).  Coherent color = a spatial bias in the posterior mean.

In [ ]:
n = len(series)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2), squeeze=False)
for a, (label, r) in zip(axes[0], series.items()):
    bias = (r['val_dead'].mean(axis=1) - r['val_truth']).mean(axis=(0, 1))
    vmax = np.abs(bias).max() + 1e-12
    im = a.imshow(bias, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    a.set_title(label, fontsize=9)
    a.set_xlabel('phi'); a.set_ylabel('eta')
    plt.colorbar(im, ax=a, fraction=0.046, label=UNIT)
fig.suptitle('spatial bias: mean(inpainted − truth) ' + UNIT)
plt.tight_layout(); plt.show()

## Diagnostic 6 — PIT histograms

Probability-integral-transform of the truth under the posterior; flat = calibrated (∪ = under-dispersed, ∩ = over-dispersed).  Rank-based, so identical in gev and log space.

In [ ]:
n = len(series)
fig, axes = plt.subplots(1, n, figsize=(3.0 * n, 3.0), squeeze=False)
for a, (label, r) in zip(axes[0], series.items()):
    pit = np.mean(r['inpaint_dead'] <= r['truth_dead'][:, None], axis=1).ravel()
    a.hist(pit, bins=20, range=(0, 1), density=True, alpha=0.8,
           color=series_style(r)['color'])
    a.axhline(1.0, color='k', ls='--')
    a.set_title(label, fontsize=9); a.set_xlabel('PIT')
axes[0][0].set_ylabel('density')
plt.tight_layout(); plt.show()

## Diagnostic 7 — Empirical coverage

Fraction of truths inside central posterior intervals; observed should track nominal.  Rank-based (space-invariant).  `meanfill` has zero posterior width, so its coverage collapses to ~0.

In [ ]:
def empirical_coverage(t, s, levels=(0.5, 0.8, 0.9, 0.95)):
    out = {}
    for level in levels:
        lo = np.quantile(s, (1 - level) / 2, axis=1)
        hi = np.quantile(s, 1 - (1 - level) / 2, axis=1)
        out[level] = float(((t >= lo) & (t <= hi)).mean())
    return out

print('Empirical coverage (nominal -> observed):')
cov_tab = {}
for label, r in series.items():
    cov = empirical_coverage(r['truth_dead'], r['inpaint_dead'])
    cov_tab[label] = cov
    print(f'  {label:16s}: ' + ', '.join(f'{l:.0%}->{o:.1%}'
                                         for l, o in cov.items()))

## Diagnostic 8 — CRPS and sharpness (sorted estimator)

CRPS (lower = sharper *and* calibrated) and mean ensemble std (sharpness), in `SPACE`.  Grouped by box size so the DDNM-vs-floor gap is visible at each geometry.

In [ ]:
def crps_sorted(t, s):
    J = s.shape[-1]
    term1 = np.abs(s - t[..., None]).mean(axis=-1)
    ss = np.sort(s, axis=-1)
    i  = np.arange(1, J + 1)
    term2 = (2 / (J * J)) * ((2 * i - J - 1) * ss).sum(axis=-1)
    return term1 - 0.5 * term2

crps_by, sharp_by = {}, {}
for label, r in series.items():
    s = np.moveaxis(r['val_dead'], 1, -1)
    crps_by[label]  = float(crps_sorted(r['val_truth'], s).mean())
    sharp_by[label] = float(r['val_dead'].std(axis=1).mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(max(9, 1.1 * len(series)), 3.8))
xs = np.arange(len(series))
cols = [series_style(series[l])['color'] for l in series]
a1.bar(xs, [crps_by[l] for l in series], color=cols)
a1.set_xticks(xs); a1.set_xticklabels(list(series), rotation=45, ha='right',
                                      fontsize=7)
a1.set_ylabel('mean CRPS ' + UNIT); a1.set_title('CRPS (lower = better)')
a2.bar(xs, [sharp_by[l] for l in series], color=cols)
a2.set_xticks(xs); a2.set_xticklabels(list(series), rotation=45, ha='right',
                                      fontsize=7)
a2.set_ylabel('mean ensemble std ' + UNIT); a2.set_title('sharpness')
plt.tight_layout(); plt.show()
for label in series:
    print(f'  {label:16s}: CRPS={crps_by[label]:.5f}  '
          f'sharpness={sharp_by[label]:.5f}')

## Diagnostic 9 — Example reconstructions

For one fixed event: truth, posterior mean, one posterior sample, and the mean−truth residual, per series (log color scale).  The green box marks the inpainted dead region.

In [ ]:
EVENT_IDX = 0
for label, r in series.items():
    i_img = r['idx'][EVENT_IDX]
    Y0, X0, B = r['Y0'], r['X0'], r['BOX']
    truth = true_events[i_img].astype(np.float64)
    mean_img = truth.copy()
    mean_img[Y0:Y0+B, X0:X0+B] = r['inpaint_dead'][EVENT_IDX].mean(axis=0)[0]
    one_img = truth.copy()
    one_img[Y0:Y0+B, X0:X0+B] = r['inpaint_dead'][EVENT_IDX, 0, 0]
    resid = mean_img - truth

    fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
    for a, (img, ttl) in zip(ax, [(truth, 'truth'), (mean_img, 'ensemble mean'),
                                  (one_img, 'one sample'), (resid, 'mean − truth')]):
        if ttl == 'mean − truth':
            v = np.abs(img).max() + 1e-9
            im = a.imshow(img, cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
        else:
            im = a.imshow(np.clip(img, 1e-3, None),
                          norm=MplLogNorm(1e-3, max(img.max(), 1e-2)),
                          cmap='viridis', aspect='auto')
        a.add_patch(plt.Rectangle((X0-.5, Y0-.5), B, B, fill=False,
                                  edgecolor='lime', lw=1.5))
        a.set_title(ttl, fontsize=10)
        plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(f'{label} — event {i_img} (green = inpainted region)')
    plt.tight_layout(); plt.show()

# Part B — Shrinkage study, box sizes

Per-image dead-box mean: truth vs posterior mean, regression per series (in `SPACE`).  Forward slope < 1 is posterior-mean shrinkage (expected, stronger for larger, less-constrained boxes); reverse slope ≈ 1 is calibration of the posterior mean as a point estimator.  fwd·rev = R².

In [ ]:
n = len(series)
fig, axes = plt.subplots(2, n, figsize=(4.0 * n, 8), squeeze=False)
shrink = {}
for col, (label, r) in enumerate(series.items()):
    t_m = r['val_truth'].mean(axis=(1, 2, 3))
    f_m = r['val_dead'].mean(axis=(1, 2, 3, 4))
    f_s = r['val_dead'].mean(axis=(2, 3, 4)).std(axis=1, ddof=1)
    st = series_style(r)

    # degenerate (zero-width, e.g. meanfill) -> reverse fit undefined
    try:
        res_f = stats.linregress(t_m, f_m)
    except ValueError:
        res_f = None
    try:
        res_r = stats.linregress(f_m, t_m)
    except ValueError:
        res_r = None
    shrink[label] = (res_f, res_r)
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]

    a = axes[0][col]
    a.errorbar(t_m, f_m, yerr=f_s, fmt='o', ms=3, alpha=0.5, capsize=2,
               color=st['color'])
    if res_f is not None:
        xx = np.linspace(t_m.min(), t_m.max(), 50)
        a.plot(xx, res_f.intercept + res_f.slope*xx, 'k-', lw=2,
               label=f'y={res_f.slope:.2f}x+{res_f.intercept:.2f}\n'
                     f'R²={res_f.rvalue**2:.2f}')
    a.plot(lim, lim, 'r--', lw=1, label='y = x')
    a.set_title(label, fontsize=9); a.set_xlabel('truth box mean ' + UNIT)
    a.grid(alpha=0.3); a.legend(fontsize=8)

    a = axes[1][col]
    a.errorbar(f_m, t_m, xerr=f_s, fmt='s', ms=3, alpha=0.5, capsize=2,
               color=st['color'])
    if res_r is not None:
        xx = np.linspace(f_m.min(), f_m.max(), 50)
        a.plot(xx, res_r.intercept + res_r.slope*xx, 'k-', lw=2,
               label=f'y={res_r.slope:.2f}x+{res_r.intercept:.2f}\n'
                     f'R²={res_r.rvalue**2:.2f}')
    a.plot(lim, lim, 'r--', lw=1, label='y = x')
    a.set_xlabel('posterior-mean box mean ' + UNIT)
    a.grid(alpha=0.3); a.legend(fontsize=8)

axes[0][0].set_ylabel('posterior-mean box mean ' + UNIT)
axes[1][0].set_ylabel('truth box mean ' + UNIT)
plt.tight_layout(); plt.show()

print('Shrinkage summary (box means, ' + SPACE + '):')
print('  forward slope: fill~truth (shrinkage, <1 expected);')
print('  reverse slope: truth~fill (calibration: ~1 for an unbiased mean)')
for label, (rf, rr) in shrink.items():
    if rf is None or rr is None:
        print(f'  {label:16s}: degenerate (zero-width posterior)')
        continue
    print(f'  {label:16s}: fwd={rf.slope:.3f}  rev={rr.slope:.3f}  '
          f'R²={rf.rvalue**2:.3f}  (fwd*rev={rf.slope*rr.slope:.3f})')

In [ ]:
# 2D-histogram view of the same forward relationship
n = len(series)
fig, axes = plt.subplots(1, n, figsize=(3.8 * n, 3.6), squeeze=False)
for a, (label, r) in zip(axes[0], series.items()):
    t_m = r['val_truth'].mean(axis=(1, 2, 3))
    f_m = r['val_dead'].mean(axis=(1, 2, 3, 4))
    h = a.hist2d(t_m, f_m, bins=15, cmap='viridis')
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]
    a.plot(lim, lim, 'r--', lw=1)
    a.set_title(label, fontsize=9); a.set_xlabel('truth box mean ' + UNIT)
    plt.colorbar(h[3], ax=a, fraction=0.046)
axes[0][0].set_ylabel('posterior-mean box mean ' + UNIT)
plt.tight_layout(); plt.show()

# Part C — Scalar metric-vs-box sweep

The scalar summary from `calo_sweep_analysis.ipynb`, restricted to cent4.  Metrics come from `calo_inpaint.analysis.summarize_run` (progress-truncated, NaN-guarded; value metrics in log space by default here, independent of the `SPACE` above).  Reference lines: pull std → finite-J calibrated √((1+1/J)(J−1)/(J−3)); coverage → nominal; bias → 0; reverse slope → 1.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('..'))
from calo_inpaint.analysis import collect_runs, pull_std_reference

SWEEP_SPACE = os.environ.get('CALO_SWEEP_SPACE', 'log')
sweep = collect_runs(STUDY_GLOB, space=SWEEP_SPACE)
sweep = [r for r in sweep if r['algorithm'] in METHODS]
print(f'{len(sweep)} cent4 runs summarized (space={SWEEP_SPACE})\n')

hdr = (f'{"alg":9s} {"box":>3s} {"n":>5s} {"pull_std":>8s} {"(ref)":>6s} '
       f'{"cov50":>6s} {"cov90":>6s} {"sbc_p":>8s} {"bias":>8s} '
       f'{"crps":>7s} {"fwd":>6s} {"rev":>6s}')
print(hdr); print('-' * len(hdr))
for r in sweep:
    print(f"{r['algorithm']:9s} {r['box']:3d} {r['n_images']:5d} "
          f"{r['pull_std']:8.3f} {r['pull_std_ref']:6.3f} "
          f"{r['cov50']:6.3f} {r['cov90']:6.3f} {r['sbc_p']:8.2g} "
          f"{r['bias_mean']:8.4f} {r['crps']:7.4f} "
          f"{r['slope_fwd']:6.3f} {r['slope_rev']:6.3f}")

In [ ]:
def sweep_lines(metric, ylabel, ref=None):
    plt.figure(figsize=(6.5, 4))
    by_alg = {}
    for r in sweep:
        by_alg.setdefault(r['algorithm'], []).append(r)
    for alg, rs in sorted(by_alg.items()):
        rs = sorted(rs, key=lambda r: r['box'])
        ls, marker = METHOD_STYLE.get(alg, ('-', 'o'))
        plt.plot([r['box'] for r in rs], [r[metric] for r in rs],
                 ls=ls, marker=marker, ms=5, label=alg)
    if ref is not None:
        plt.axhline(ref, color='k', ls=':', lw=1, label='calibrated')
    plt.xlabel('box size [towers]'); plt.ylabel(ylabel)
    plt.grid(alpha=0.3); plt.legend(fontsize=8)
    plt.tight_layout(); plt.show()

J_ref = sweep[0]['n_samples'] if sweep else 50
sweep_lines('pull_std', 'pull std (log space)', ref=pull_std_reference(J_ref))
sweep_lines('cov90', 'empirical coverage @ 90%', ref=0.90)
sweep_lines('cov50', 'empirical coverage @ 50%', ref=0.50)
sweep_lines('crps', 'mean CRPS (log space)')
sweep_lines('bias_mean', 'mean bias (log space)', ref=0.0)
sweep_lines('slope_rev', 'reverse slope (truth ~ posterior mean)', ref=1.0)

## Baselines as floors

How much of DDNM's CRPS advantage over "fill with known-region statistics" survives at each box size.

In [ ]:
boxes_s = sorted({r['box'] for r in sweep})
present = [m for m in ('ddnm', 'noise', 'meanfill')
           if any(r['algorithm'] == m for r in sweep)]
if boxes_s and present:
    print(f'{"box":>4s}  ' + ''.join(f'{a:>12s}' for a in present)
          + '   (CRPS, log space; lower = better)')
    for b in boxes_s:
        vals = []
        for alg in present:
            r = next((r for r in sweep
                      if r['algorithm'] == alg and r['box'] == b), None)
            vals.append(f"{r['crps']:12.4f}" if r else f'{"-":>12s}')
        print(f'{b:4d}  ' + ''.join(vals))
else:
    print('no baseline runs found alongside DDNM')